# Phase 4 — Market Microstructure & Order Flow Research Notebook

This interactive research notebook executes the complete **Phase 4 Microstructure Research Workflow**:
1. Ingestion of high-frequency **aggTrades**, **Open Interest (OI)**, and **Liquidation Events**.
2. Vectorized computation of **Cumulative Volume Delta (CVD)**, Session CVD, and Divergence primitives.
3. Classification of the **4-State Joint Price / Open Interest Regime Matrix**.
4. Counterfactual shadow analysis and **Microstructure Feature Ablation** on Champion trades.

In [ ]:
# =============================================================================
# 0. GOOGLE COLAB / LOCAL REPOSITORY SYNC & SETUP
# =============================================================================
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/umutergul74/daytrader.git"
REPO_DIR = Path("/content/daytrader")

if IN_COLAB:
    print("🚀 [Google Colab Detected] Initializing Daytrader Platform...")
    if not REPO_DIR.exists():
        print(f"Cloning latest repository from {REPO_URL}...")
        !git clone {REPO_URL} /content/daytrader
    else:
        print("Pulling latest updates from GitHub...")
        !cd /content/daytrader && git pull

    os.chdir(str(REPO_DIR))
    print("Installing dependencies...")
    !pip install -q polars pandas numpy scipy scikit-learn lightgbm xgboost catboost pydantic pydantic-settings typer rich matplotlib pyarrow requests websockets pytest optuna

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    print(f"✓ Environment ready! Working directory: {Path.cwd()}")
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(str(project_root))
    if str(project_root / "src") not in sys.path:
        sys.path.insert(0, str(project_root / "src"))
    print(f"✓ [Local Mode] Working directory: {Path.cwd()}")

import polars as pl
import numpy as np
from quant_platform.data.storage.canonical import CanonicalStorage
from quant_platform.data.storage.event_storage import CanonicalEventStorage
from quant_platform.data.providers.microstructure import BinanceAggTradesProvider, BinanceMicrostructureProvider
from quant_platform.features.microstructure.cvd import CvdEngine
from quant_platform.features.microstructure.trade_flow import TradeFlowEngine
from quant_platform.features.microstructure.open_interest_features import OpenInterestFeatureEngine
from quant_platform.features.microstructure.liquidation_features import LiquidationFeatureEngine
from quant_platform.research.counterfactual_analyzer import CounterfactualAnalyzer
from quant_platform.research.microstructure_ablation import MicrostructureAblationEngine

print("✓ Microstructure research environment initialized.")

## 1. Load Canonical Data & Generate Aligned Microstructure Events

In [ ]:
storage = CanonicalStorage()
df_1m = storage.read_symbol("ETHUSDT", start_year=2024, start_month=1).head(1000)

# Synthesize/load aligned aggTrades, OI, and Liquidations
df_trades = BinanceAggTradesProvider.generate_synthetic_agg_trades(df_1m, trades_per_bar=10)
df_oi = BinanceMicrostructureProvider.generate_synthetic_open_interest(df_1m)
df_liqs = BinanceMicrostructureProvider.generate_synthetic_liquidations(df_1m)

print(f"Loaded {len(df_1m)} 1m bars, {len(df_trades)} aggTrades, {len(df_oi)} OI snapshots, {len(df_liqs)} liquidations.")

## 2. Compute CVD, Trade Flow, Open Interest & Liquidation Features

In [ ]:
# 1. CVD and Delta features
df_feat = CvdEngine.compute_kline_delta_features(df_1m, rolling_window=20)
df_feat = CvdEngine.detect_cvd_divergence(df_feat, lookback=5)

# 2. Trade flow imbalance
df_feat = TradeFlowEngine.compute_trade_flow_features(df_feat, imbalance_window=14)

# 3. Open Interest alignment & 4-State regime
df_feat = OpenInterestFeatureEngine.align_and_compute_oi_features(df_feat, df_oi)

# 4. Liquidation features
df_feat = LiquidationFeatureEngine.align_and_compute_liquidation_features(df_feat, df_liqs)

print("Computed microstructure features:")
print(df_feat.select(["close_time", "close", "delta_1m", "cvd_global", "joint_price_oi_regime", "is_liquidation_burst"]).tail(5))

## 3. Microstructure Counterfactual Analysis & Incremental Edge Gate

In [ ]:
mock_trades = [
    {"net_pnl": 100.0, "r_multiple": 2.0, "direction": "LONG", "cvd_bullish_divergence": True, "joint_price_oi_regime": "LONG_BUILDUP", "trade_imbalance_1m": 0.20, "is_liquidation_burst": True},
    {"net_pnl": -50.0, "r_multiple": -1.0, "direction": "LONG", "cvd_bullish_divergence": False, "joint_price_oi_regime": "LONG_LIQUIDATION", "trade_imbalance_1m": -0.10, "is_liquidation_burst": False},
    {"net_pnl": 80.0, "r_multiple": 1.6, "direction": "SHORT", "cvd_bearish_divergence": True, "joint_price_oi_regime": "SHORT_BUILDUP", "trade_imbalance_1m": -0.15, "is_liquidation_burst": True},
    {"net_pnl": -50.0, "r_multiple": -1.0, "direction": "SHORT", "cvd_bearish_divergence": False, "joint_price_oi_regime": "SHORT_COVERING", "trade_imbalance_1m": 0.10, "is_liquidation_burst": False},
] * 10

study = CounterfactualAnalyzer.run_standard_counterfactual_suite("smc:liquidity_sweep_fvg:v2", mock_trades)

for f_res in study.filter_evaluations:
    print(f"[{f_res.marginal_edge_status}] {f_res.filter_name}: Trades={f_res.filtered_trade_count} (-{f_res.trade_reduction_pct:.0f}%), WR={f_res.filtered_win_rate:.1f}%, PF={f_res.filtered_profit_factor:.2f}, ExpDelta={f_res.expectancy_delta_r:+.2f}R")